In [11]:
# Step 1: Import libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
# Step 2: Define transformations (resize + normalize)
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))])


dataset_root = '/content/drive/My Drive/dataset'

train_data = datasets.ImageFolder(root=dataset_root, transform=transform)
test_data = datasets.ImageFolder(root=dataset_root, transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [3]:
# Step 4: Define baseline CNN
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(64*32*32, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64*32*32)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

num_classes = len(train_data.classes)
model = SimpleCNN(num_classes)
print(train_data.classes)

['Bear', 'Bird', 'Cat', 'Cow', 'Deer', 'Dog', 'Dolphin', 'Elephant', 'Giraffe', 'Horse', 'Kangaroo', 'Lion', 'Panda', 'Tiger', 'Zebra']


In [4]:

# Step 5: Define loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [6]:
# Step 6: Training loop
epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

    # Step 7: Evaluation
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Epoch 1, Loss: 2.594414550749982
Epoch 2, Loss: 2.1655241133736784
Epoch 3, Loss: 1.6496713904083753
Epoch 4, Loss: 1.129869342827406
Epoch 5, Loss: 0.787461162102027
Epoch 6, Loss: 0.5352830466676931
Epoch 7, Loss: 0.5246242360501993
Epoch 8, Loss: 0.43151056180234815
Epoch 9, Loss: 0.3680127645369436
Epoch 10, Loss: 0.2527299497093334
Test Accuracy: 94.55%


In [7]:
from sklearn.metrics import classification_report

# Step 7 (extended): Evaluation with metrics
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1) # Extract only the predicted indices
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Precision, Recall, F1
print("Classification Report:\n")
print(classification_report(all_labels, all_preds, target_names=train_data.classes))


Classification Report:

              precision    recall  f1-score   support

        Bear       0.98      0.96      0.97       125
        Bird       1.00      0.93      0.96       137
         Cat       0.93      0.94      0.94       123
         Cow       0.96      0.99      0.97       131
        Deer       0.94      0.91      0.93       127
         Dog       0.68      0.99      0.81       122
     Dolphin       0.99      0.99      0.99       129
    Elephant       0.93      0.98      0.96       133
     Giraffe       0.95      0.98      0.96       129
       Horse       1.00      0.90      0.95       130
    Kangaroo       0.98      0.86      0.92       126
        Lion       0.99      0.99      0.99       131
       Panda       0.99      0.99      0.99       135
       Tiger       0.98      0.91      0.94       129
       Zebra       1.00      0.85      0.92       137

    accuracy                           0.95      1944
   macro avg       0.95      0.95      0.95      1944
we

In [10]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

def evaluate_model(model, test_loader, class_names):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Compute metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='macro')
    recall = recall_score(all_labels, all_preds, average='macro')
    f1 = f1_score(all_labels, all_preds, average='macro')

    return accuracy, precision, recall, f1

results = []
baseline_model = model

for model_name, current_model in [("Baseline CNN", baseline_model)]:
    acc, prec, rec, f1 = evaluate_model(current_model, test_loader, train_data.classes)
    results.append([model_name, acc, prec, rec, f1])

df = pd.DataFrame(results, columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score"])
print(df)


          Model  Accuracy  Precision    Recall  F1 Score
0  Baseline CNN  0.945473   0.953448  0.945647  0.946448
